<a href="https://colab.research.google.com/github/nelsonlam23/iap/blob/main/Another_copy_of_iAPI_Deferred_Model_Call_Fishfooding_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Copyright 2026 Google LLC.

In [ ]:
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Colab Summary: Deferred Tier Model Call

This colab shows how to make a **deferred** (asynchronous) model call to a
Gemini text model on Vertex through the Interaction API SDK. A deferred model
call is just a regular model call with **`service_tier="deferred"`**.

The deferred tier submits your request to a background queue and returns an
**Interaction ID immediately**. The request is processed on flexible, off-peak
capacity, and you retrieve the result later by polling with the Interaction ID.

**What is different from a regular model call:**
*   `service_tier="deferred"`  (this is the only functional difference)
*   The project must be **allowlisted** for deferred interaction scheduling.
    For this fishfood use **`cloud-llm-preview2`** (project number `323656405210`).

**Deferred requirements / constraints:**
*   `service_tier="deferred"`
*   `background=True`   (deferred runs in the background)
*   `store=True`        (result must be stored to fetch it later)
*   `stream=False`      (streaming is not used with background/deferred)

**Allowlisted projects** (see `EnableDeferredInteractionScheduling` Mendel flag):
*   `cloud-llm-preview2` (Project number: `323656405210`)  <- use this one

**Available models:**
*   gemini-3.5-flash
*   gemini-3.6-flash
*   gemini-3.7-flash
*   gemini-3.1-flash-lite
*   gemini-3.5-flash-lite
*   gemini-3.1-pro-preview

# SDK Setup

In [ ]:
# Version note (avoids pip dependency conflicts on the Colab runtime):
#   * google-adk (preinstalled) needs google-genai >=2.12.1,<3
#   * google-colab (preinstalled) pins google-auth==2.49.0
#   * google-genai 2.14+ requires google-auth>=2.56.0, which conflicts with that
#     google-auth pin.
# google-genai==2.12.1 is the version that satisfies BOTH, so it installs with
# no dependency conflicts. It usually matches what the runtime already ships
# (so this is often a no-op). The "deferred" service tier is passed as a plain
# string below and is accepted by this SDK version (forward-compatible enum),
# then sent as-is to the API.
!pip3 install -q "google-genai==2.12.1"
!pip show google-genai
# NOTE: if you already imported google.genai in this runtime, use
# Runtime > Restart session before running the cells below.
#
# Prefer the newest SDK with a first-class "deferred" enum? Use
#   !pip3 install -q "google-genai>=2.18.0"
# but you will then see a (harmless) google-colab/google-auth==2.49.0 warning,
# because 2.18+ pulls google-auth>=2.56.0. authenticate_user() still works.

# Environment and Authentication Setup

In [ ]:
LOCATION = "global"
ENVIRONMENT = "prod" # @param ["autopush","staging","prod"] {"allow-input":true}

# @markdown ### ⚙️ API Configuration
# @markdown Use an allowlisted project. For this fishfood: cloud-llm-preview2.
PROJECT_NUMBER = "cloud-llm-preview4"  # @param {type:"string"}

from google.colab import auth
auth.authenticate_user(project_id=PROJECT_NUMBER)
import os
import time
import subprocess
import requests
from google import genai
from google.genai import types

if ENVIRONMENT == "autopush":
  BASE_URL = 'https://autopush-aiplatform.sandbox.googleapis.com'
elif ENVIRONMENT == "staging":
  BASE_URL = 'https://staging-aiplatform.sandbox.googleapis.com'
elif ENVIRONMENT == "prod":
  BASE_URL = 'https://aiplatform.googleapis.com'
else:
  raise ValueError(f"Invalid environment: {ENVIRONMENT}")

os.environ['GOOGLE_VERTEX_BASE_URL'] = BASE_URL
INTERACTIONS_PATH = BASE_URL + f"/v1beta1/projects/{PROJECT_NUMBER}/locations/{LOCATION}/interactions"

# Construct the client.
client = genai.Client(
    vertexai=True,
    project=PROJECT_NUMBER,
    location=LOCATION,
)

def _get_headers():
  token = subprocess.check_output(["gcloud", "auth", "print-access-token"]).decode("utf-8").strip()
  return {
      "Authorization": f"Bearer {token}",
      "Content-Type": "application/json",
      "X-Goog-User-Project": PROJECT_NUMBER,
  }

print(f"Environment : {ENVIRONMENT}")
print(f"Base URL    : {BASE_URL}")
print(f"Project     : {PROJECT_NUMBER}")

# 1. Create a Deferred Model Call

An **`interaction_id`** is returned immediately. The request is queued and runs
in the background; you retrieve the result later (Section 2) using this ID.

The only change from a regular model call is `service_tier="deferred"` (plus the
required `background=True`, `store=True`, `stream=False`).

In [ ]:
# @markdown ### ⚙️ Create Deferred Interaction
user_input = "What is the population of the capital of China?"  # @param {type:"string"}
MODEL = "gemini-3.5-flash" # @param ["gemini-3.5-flash","gemini-3.6-flash","gemini-3.7-flash","gemini-3.1-flash-lite","gemini-3.5-flash-lite","gemini-3.1-pro-preview"]

# Deferred tier requires: background=True, store=True, stream=False.
response = client.interactions.create(
    model=MODEL,
    input=user_input,
    service_tier="deferred",
    background=True,
    store=True,
    stream=False,
)

print(response)
CREATED_INTERACTION_ID = getattr(response, "id", None)
print("\nInteraction ID :", CREATED_INTERACTION_ID)
print("Status         :", getattr(response, "status", None))
print("Service tier   :", getattr(response, "service_tier", None))
print("\n→ Copy the Interaction ID above into the next cell to poll for the result.")

status='queued' model='gemini-3.5-flash' agent=None id='ChA3MTM3ZjBjMGY3M2NjNjI0EAgaATAqBG1haW4' created=None updated=None system_instruction=None tools=None usage=None response_modalities=None response_mime_type=None previous_interaction_id=None environment_id=None service_tier=None webhook_config=None steps=None response_format=None environment=None generation_config=None agent_config=None safety_settings=None labels=None input=None output_text='' output_image=None output_audio=None output_video=None object='interaction'

Interaction ID : ChA3MTM3ZjBjMGY3M2NjNjI0EAgaATAqBG1haW4
Status         : queued
Service tier   : None

→ Copy the Interaction ID above into the next cell to poll for the result.


# 2. Poll & Get the Result

The deferred request runs asynchronously. Poll `GetInteraction` with the
Interaction ID until `status` becomes `completed` (or `failed` / `cancelled`).
This cell uses a direct REST `GET` so the raw JSON is unambiguous, then prints
the **user input** and the **model output**.

A stored deferred interaction returns the transcript in `steps[]` (with step
types `user_input`, `thought`, and `model_output`); the model's answer may be
split across several `model_output` steps, so we concatenate their text and skip
`thought` steps.

In [ ]:
# @markdown ### ⚙️ Poll for Status / Result
interaction_id = ""  # @param {type:"string"}
POLL_INTERVAL_S = 15  # @param {type:"integer"}
MAX_WAIT_S = 1800  # @param {type:"integer"}

if not interaction_id:
  interaction_id = CREATED_INTERACTION_ID
  print(f"Using Interaction ID from previous cell: {interaction_id}")

def _collect_text(content):
  # Join the text parts of a step/output; skip non-text parts (e.g. thoughts).
  return "".join(p.get("text", "") for p in (content or [])
                 if p.get("type") == "text")

def show_interaction(data):
  usage = data.get("usage", {}) or {}
  print("\n--- Final interaction ---")
  print("id           :", data.get("id"))
  print("status       :", data.get("status"))
  print("service_tier :", data.get("service_tier"), " <-- expect 'deferred'")
  print("model        :", data.get("model"))
  print("usage        : total={} input={} output={} thought={}".format(
      usage.get("total_tokens"), usage.get("total_input_tokens"),
      usage.get("total_output_tokens"), usage.get("total_thought_tokens")))

  user_parts, output_parts = [], []
  # Stored/deferred responses carry the transcript in steps[].
  for step in data.get("steps", []) or []:
    stype = step.get("type")
    if stype == "user_input":
      user_parts.append(_collect_text(step.get("content")))
    elif stype == "model_output":
      output_parts.append(_collect_text(step.get("content")))
    # 'thought' steps are intentionally skipped.
  # Fallback for the streamed/unary shape that uses outputs[].
  for out in data.get("outputs", []) or []:
    if out.get("type") == "text":
      output_parts.append(out.get("text", ""))

  user_text = "".join(user_parts).strip()
  model_text = "".join(output_parts).strip()
  print("\n=== User input ===\n" + (user_text or "(none found)"))
  print("\n=== Model output ===\n" + (model_text or "(no text output found)"))

def poll_interaction(iid):
  # Poll GetInteraction until a terminal status, an API error, or timeout.
  # Prints progress and the final user input / model output.
  # Returns (data, error). Reused by the follow-up cell in Section 3.
  get_url = f"{INTERACTIONS_PATH}/{iid}?stream=false"
  start = time.time()
  data = {}
  error = None
  while True:
    resp = requests.get(get_url, headers=_get_headers())
    data = resp.json()
    # Fail fast on an API error (e.g. a deleted / not-found interaction) or a
    # non-200 response, instead of polling until timeout.
    error = data.get("error") if isinstance(data, dict) else None
    if error or resp.status_code != 200:
      error = error or {"code": resp.status_code, "message": str(data)}
      print(f"Request failed: code={error.get('code')} message={error.get('message')}")
      break
    status = data.get("status")
    elapsed = int(time.time() - start)
    print(f"[{elapsed:4d}s] status={status} service_tier={data.get('service_tier')}")
    if status in ("completed", "failed", "cancelled"):
      break
    if elapsed > MAX_WAIT_S:
      print("Timed out waiting; interaction is still in progress. Re-run to keep polling.")
      break
    time.sleep(POLL_INTERVAL_S)
  if not error:
    show_interaction(data)
  return data, error

poll_interaction(interaction_id)

Using Interaction ID from previous cell: ChA3MTM3ZjBjMGY3M2NjNjI0EAgaATAqBG1haW4
[   1s] status=queued service_tier=deferred
[  18s] status=queued service_tier=deferred
[  35s] status=queued service_tier=deferred
[  52s] status=queued service_tier=deferred
[  69s] status=in_progress service_tier=deferred
[  87s] status=completed service_tier=deferred

--- Final interaction ---
id           : ChA3MTM3ZjBjMGY3M2NjNjI0EAgaATAqBG1haW4
status       : completed
service_tier : deferred  <-- expect 'deferred'
model        : gemini-3.5-flash
usage        : total=278 input=10 output=39 thought=229

=== User input ===
What is the population of the capital of China?

=== Model output ===
The capital of China is **Beijing**. 

As of the end of 2023, the permanent resident population of Beijing was approximately **21.84 million** people.


({'id': 'ChA3MTM3ZjBjMGY3M2NjNjI0EAgaATAqBG1haW4',
  'status': 'completed',
  'usage': {'total_tokens': 278,
   'total_input_tokens': 10,
   'input_tokens_by_modality': [{'modality': 'text', 'tokens': 10}],
   'total_output_tokens': 39,
   'output_tokens_by_modality': [{'modality': 'text', 'tokens': 39}],
   'total_thought_tokens': 229},
  'service_tier': 'deferred',
  'steps': [{'type': 'user_input',
    'content': [{'text': 'What is the population of the capital of China?',
      'type': 'text'}]},
   {'signature': 'AY89a18kSpwUa5BLzdQtZYSlGX42iAfSov73Bn6rw9FHK87DFYa13+BFfRVmd69n/c9LPs89CdqzOBoPs2XauzYcxI48a7ByHuc87rKAVkt+AKgksc+tFZjlEw2vV6uG+6ggxuaVVNfcpcdnt4/58F3ZZC83iWFBazqEo5DqC443OnqzPRo9Jdhzq1y7/5zGH00c4/MTFGBwdRIciRi5JvnTIsd1maeaRBF6YvqhtJ8SL/na5Dx3cPuQXrtanLYSJgVS4syC0qOK6uT0TP6Xp7l/u6n/dzheqAM+JmH+s3f32Wg/OkTkHagAA+VEXD8wfd8xvrt7GXyD9abR4kVRzFjcimqsrM9LLbKimVsYZmwtm7vPXN61O0Wx1Ptl0AzFGY6atGTXcueigs/4MhqIbpBDsa2X8xNYWfJXV18mJGU1Sqisq97TZXxkzgY2biYQ23j4wvlRAX8QStVyN1O2Db58PPqa

### (Optional) Get via SDK

The same retrieval using the SDK instead of REST.

In [ ]:
interaction_id = interaction_id or CREATED_INTERACTION_ID
result = client.interactions.get(id=interaction_id)
print(result)

# 3. Follow-up Turn (previous_interaction_id)

Chain a second deferred model call to the first by passing
`previous_interaction_id`. The earlier interaction must be **completed** and
stored (`store=True`) for its context to carry over, so run Sections 1 and 2
first. This cell reuses `poll_interaction(...)` defined in Section 2.

In [ ]:
# @markdown ### ⚙️ Follow-up Turn (previous_interaction_id)
previous_interaction_id = ""  # @param {type:"string"}
followup_input = "How does that compare to the population of Shanghai?"  # @param {type:"string"}
MODEL = "gemini-3.5-flash" # @param ["gemini-3.5-flash","gemini-3.6-flash","gemini-3.7-flash","gemini-3.1-flash-lite","gemini-3.5-flash-lite","gemini-3.1-pro-preview"]

# Default to the interaction created in Section 1. It must be completed first.
previous_interaction_id = previous_interaction_id or CREATED_INTERACTION_ID
print(f"Chaining from previous interaction: {previous_interaction_id}")

try:
  followup = client.interactions.create(
      model=MODEL,
      input=followup_input,
      previous_interaction_id=previous_interaction_id,
      service_tier="deferred",
      background=True,
      store=True,
      stream=False,
  )
except Exception as e:
  # A common cause is the previous interaction not being completed yet.
  print(f"Follow-up create failed: {getattr(e, 'message', None) or e}")
  followup = None

if followup is not None:
  FOLLOWUP_INTERACTION_ID = getattr(followup, "id", None)
  print("Follow-up Interaction ID :", FOLLOWUP_INTERACTION_ID)
  print("Status                   :", getattr(followup, "status", None))
  # Wait for the follow-up result and display it (reuses the Section 2 helper).
  poll_interaction(FOLLOWUP_INTERACTION_ID)

Chaining from previous interaction: ChA3MTM3ZjBjMGY3M2NjNjI0EAgaATAqBG1haW4
Follow-up Interaction ID : ChA4MmIwZjUwMDY2MDdmZjMyEAgaATAqBG1haW4
Status                   : queued
[   1s] status=queued service_tier=deferred
[  18s] status=queued service_tier=deferred
[  36s] status=queued service_tier=deferred
[  53s] status=queued service_tier=deferred
[  73s] status=in_progress service_tier=deferred
[  90s] status=in_progress service_tier=deferred
[ 109s] status=in_progress service_tier=deferred
[ 127s] status=in_progress service_tier=deferred
[ 144s] status=in_progress service_tier=deferred
[ 162s] status=in_progress service_tier=deferred
[ 180s] status=in_progress service_tier=deferred
[ 197s] status=in_progress service_tier=deferred
[ 215s] status=in_progress service_tier=deferred
[ 234s] status=in_progress service_tier=deferred
[ 251s] status=in_progress service_tier=deferred
[ 269s] status=in_progress service_tier=deferred
[ 287s] status=completed service_tier=deferred

--- Final i

# 4. List / Cancel / Delete Interactions

In [ ]:
# @markdown ### ⚙️ List Interactions
API_REVISION = "2026-05-20"
next_page_token = ""  # @param {type:"string"}

headers = _get_headers()
headers["Api-Revision"] = API_REVISION
resp = requests.get(INTERACTIONS_PATH, headers=headers,
                    params={"pageSize": 10, "pageToken": next_page_token})
data = resp.json()
print("Interactions   :", data.get("interaction_metadatas", []))
print("Next Page Token:", data.get("next_page_token"))

In [ ]:
# @markdown ### ⚙️ Cancel Interaction
interaction_id = ""  # @param {type:"string"}
interaction_id = interaction_id or CREATED_INTERACTION_ID

# Cancel is only valid while the interaction is still in progress. If it has
# already completed/failed/cancelled (409) or does not exist (404), the SDK
# raises; handle it gracefully instead of crashing the cell.
try:
  response = client.interactions.cancel(id=interaction_id)
  print("Cancelled:", response)
except Exception as e:
  print(f"Cancel failed: {getattr(e, 'message', None) or e}")

In [ ]:
# @markdown ### ⚙️ Delete Interaction
interaction_id = ""  # @param {type:"string"}
interaction_id = interaction_id or CREATED_INTERACTION_ID

# Delete can fail if the interaction is still in progress (409) or does not
# exist (404); handle it gracefully instead of crashing the cell.
try:
  response = client.interactions.delete(id=interaction_id)
  print("Deleted:", response)
except Exception as e:
  print(f"Delete failed: {getattr(e, 'message', None) or e}")

# 5. Reference: `curl` equivalents

The same deferred model call over REST, for fishfooders who prefer the command
line. Run these in a terminal.

**Autopush**
```
gcloud auth application-default login
TOKEN=$(gcloud auth application-default print-access-token)
PROJECT_ID=cloud-llm-preview4
PROMPT="What is the population of the capital of China?"

# 1. Kick off the deferred model call (returns quickly with an Interaction ID).
curl -X POST \
  -H "Authorization: Bearer ${TOKEN}" \
  -H "Content-Type: application/json" \
  -H "X-Goog-User-Project: ${PROJECT_ID}" \
  "https://autopush-aiplatform.sandbox.googleapis.com/v1beta1/projects/${PROJECT_ID}/locations/global/interactions" \
  -d "{
    \"stream\": false,
    \"background\": true,
    \"store\": true,
    \"service_tier\": \"deferred\",
    \"model\": \"gemini-3.5-flash\",
    \"input\": [ {\"type\": \"text\", \"text\": \"${PROMPT}\"} ]
  }"

# 2. Poll for status/result. Replace with the returned id.
INTERACTION_ID="<INTERACTION_ID_FROM_ABOVE>"
curl -X GET \
  -H "Authorization: Bearer ${TOKEN}" \
  -H "X-Goog-User-Project: ${PROJECT_ID}" \
  "https://autopush-aiplatform.sandbox.googleapis.com/v1beta1/projects/${PROJECT_ID}/locations/global/interactions/${INTERACTION_ID}?stream=false"
```

**Prod** — same body, swap the host to `https://aiplatform.googleapis.com`.